In [18]:
# Detect if running in Google Colab
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if in_colab():
    # Only runs in Colab, skipped on local
    # 1. Clone the repo and set paths
    REPO_URL = "https://github.com/MadKeyboardArtist/5703-Federated-Model.git"
    REPO_DIR = "/content/5703-Federated-Model"

    import os, sys
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    os.chdir(REPO_DIR)
    sys.path.append(REPO_DIR)
    print("CWD:", os.getcwd())

    # GPU check
    import torch, subprocess, textwrap
    print("CUDA available:", torch.cuda.is_available())
    !nvidia-smi

    # update the files
    %cd /content/5703-Federated-Model
    !git pull origin main

else:
    print("Running locally — skipping Colab setup.")


Running locally — skipping Colab setup.


In [19]:
# initialize model

# Per Round:
# server copies and sends the global model weights.
# client trains locally (on their local dataset).
# client returns its new weights and sample count.
# server aggregates the weights using FedAvg.
# global model is updated.
# (Optional) Evaluate global model on a held-out test set.


# heads record:
# 1. always save the newest
# 2. always save the best ever
# 3. always save curretn best

In [20]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [21]:
# external libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import importlib.util
import json
import shutil
import pandas as pd

from collections import defaultdict

In [ ]:
# self-built files

# 1. model structure
from FederatedModel.federated_multihead_model import SharedEncoders, TabularClientModel, ImageClientModel, MultiClientModel
from FederatedModel.model_config import D_TABULAR, D_EMBEDDING, D_FUSION

# 2. site training functions
from FinalEvaluations.tabular_site_evaluation import final_evaluation as tabular_site_evaluation
from FinalEvaluations.image_site_evaluation   import final_evaluation as image_site_evaluation
# from multi_site_training   import training_loop as multi_training_loop

# 3. aggregration functions
from AggregationFunctions.aggregation_algorithms import aggregate_fedavg  as fedavg
from AggregationFunctions.aggregation_algorithms import aggregate_fedprox as fedprox
from AggregationFunctions.aggregation_algorithms import aggregate_fedadam as fedadam

In [23]:
# reimport some self-built files if any changes
import importlib
import FederatedModel.federated_multihead_model
import SiteTrainingFunctions.tabular_site_training
import SiteTrainingFunctions.image_site_training

importlib.reload(FederatedModel.federated_multihead_model)
importlib.reload(SiteTrainingFunctions.tabular_site_training)
importlib.reload(SiteTrainingFunctions.image_site_training)

<module 'SiteTrainingFunctions.image_site_training' from 'd:\\USYD\\2025 S2\\5703 Capstone\\model\\SiteTrainingFunctions\\image_site_training.py'>

In [24]:
# basic confgs initialization
# outcomes saving
SAVED_MODELS_FOLDER = "SavedModels"
DATASETS_FOLDER = "Datasets"

In [25]:
def assign_local_evaluation_function (modality):
    if modality == "tabular":
        return tabular_site_evaluation
    elif modality == "image":
        return image_site_evaluation
        # return complete_image_training
    elif modality == "multi":
        pass
    else:
        # report ERROR
        return None

In [26]:
def load_transform_from_file(tsfm_file_path):
    module_name = os.path.splitext(os.path.basename(tsfm_file_path))[0]
    spec = importlib.util.spec_from_file_location(module_name, tsfm_file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.tsfm  # assumes each .py defines a variable `transform`

In [27]:
def extract_global_components(global_state, prefix):
    # Assume global_state is a full SharedEncoders.state_dict()
    # with keys like "tabular_enc.fc1.weight", "image_enc.conv1.weight", etc.

    enc_state = {
        k: v.clone()
        for k, v in global_state.items()
        if k.startswith(prefix) # keep only parameters start with prefix
    }

    return enc_state

In [28]:
def add_prefix_to_enc_dict (prefix, state_dict_list):
    result = []
    for state in state_dict_list:
        # Add the prefix to every key
        enc_with_prefix = {
            f"{prefix}{k}": v.clone() for k, v in state.items()
        }
        result.append(enc_with_prefix)
    return result

In [29]:
def local_training (global_state, site, freeze_global = False):
    if site["modality"] == "tabular":
        site_name              = site["name"]
        train_set_path         = site["clean_dataset_train"]
        val_set_path           = site["clean_dataset_val"]
        label_col_name         = site["label_col"]
        num_of_classes         = site["n_classes"]
        newest_head_path       = site["newest_head"]
        current_best_head_path = site["current_best_head"]
        freeze_global          = freeze_global

        return site["site_training"](
            site_name              = site_name,
            global_state           = global_state,
            freeze_global          = freeze_global,
            train_set_path         = train_set_path,
            val_set_path           = val_set_path,
            labelcol               = label_col_name,
            n_classes              = num_of_classes,
            newest_head_path       = newest_head_path,
            current_best_head_path = current_best_head_path
            )

    elif site["modality"] == "image":
        site_name              = site["name"]
        train_set_path         = site["clean_dataset_train"]
        val_set_path           = site["clean_dataset_val"]
        site_tsfm              = site["tsfm"]
        num_of_classes         = site["n_classes"]
        newest_head_path       = site["newest_head"]
        current_best_head_path = site["current_best_head"]
        freeze_global          = freeze_global

        return site["site_training"](
            site_name              = site_name,
            global_state           = global_state,
            freeze_global          = freeze_global,
            train_set_path         = train_set_path,
            val_set_path           = val_set_path,
            tsfm                   = site_tsfm,
            n_classes              = num_of_classes,
            newest_head_path       = newest_head_path,
            current_best_head_path = current_best_head_path
            )

    elif site["modality"] == "multi":
        # Not implemented yet
        return None

    else:
        print("site modality error")
        exit()

In [ ]:
def complete_site_evaluation(global_state, sites):
    # 1. sever send global encoder weights to sites
    # by passing global_state

    # 2. full local training
    ################ COME TO EACH SITES ##################
    tabular_enc_update = []
    tabular_sample_count = []
    image_enc_update = []
    image_sample_count = []
    fusion_enc_update = []
    fusion_sample_count = []

    all_ckpt_eva_results = []

    for site in sites:
        # full local training
        # using all site info and correct modality training function
        updated_state, sample_count, ckpt_eva_results = local_training(global_state, site)
        ckpt_eva_results["site_name"] = site["name"]

        # record the reaults
        if site["modality"] == "tabular":
            tabular_enc_update.append(updated_state)
            tabular_sample_count.append(sample_count)
        elif site["modality"] == "image":
            image_enc_update.append(updated_state)
            image_sample_count.append(sample_count)
        elif site["modality"] == "multi":
            # fusion and all encoders
            update_tabular_enc = updated_state[0]
            update_image_enc   = updated_state[1]
            update_fusion      = updated_state[2]

            tabular_enc_update.append(update_tabular_enc)
            image_enc_update.append  (update_image_enc)
            fusion_enc_update.append (update_fusion)

            tabular_sample_count.append(sample_count)
            image_sample_count.append  (sample_count)
            fusion_sample_count.append (sample_count)
        # current_best_client_heads.append(best_head_state)
        # newest head, current best head recorded in .pth

        all_ckpt_eva_results.append(ckpt_eva_results)
    ################## END in sites, back to server ##################

    # 4. reutrns all trained global encoders from all sites
    if freeze_global:
        print("Finish head re-training")
    else:
        print("Finish site traininig")

    result = {
        "tabular_update": tabular_enc_update,   # list of model.enc.tabular_enc.state_dict()
        "tabular_count" : tabular_sample_count,
        "image_update"  : image_enc_update,     # list of model.enc.image_enc.state_dict()
        "image_count"   : image_sample_count,
        "fusion_update" : fusion_enc_update,    # list of model.enc.fusion_enc.state_dict()
        "fusion_count"  : fusion_sample_count,
    }

    return result, all_ckpt_eva_results # ((state_dict, int:sample_count), list of dict)

In [31]:
def federated_ckpt_evaluation (current_results, best_results):
    # traverse through all sites
    new_best_found = False

    # 1. calculate the results
    # macro avg acc only (best for check point)
    current_results_df = pd.DataFrame(current_results)
    macro_acc_current  = current_results_df["val_acc"].mean()
    # weighted_acc = (current_results_df["val_acc"] * current_results_df["num_samples"]).sum() / current_results_df["num_samples"].sum()

    # 2. worst-site guard: Keep it only if no site acc dropped by more than 2 %.
    worst_acc_current = current_results_df["val_acc"].min()

    best_results_df = pd.DataFrame(best_results)
    macro_acc_best  = best_results_df["val_acc"].mean()
    worst_acc_best  = best_results_df["val_acc"].min()

    if macro_acc_current > macro_acc_best and worst_acc_current >= worst_acc_best - 0.02:
        # better macro avg & no sever single drop -> keep the result
        # record best scores
        best_results = current_results # list, not _df
        # tell the caller to save best global model (encoders)
        new_best_found = True
    else:
        # new model is not better
        new_best_found = False

    # 6. return the results
    return new_best_found, best_results

In [32]:
# 3. all sites preparations:
# FOR EACH SITE:
# FOR EACH SITE:
# 3.1 DATA: get testing dataset (DONE)
# 3.2 MODEL: find the head path (with site name)
# 3.4 assign the correct local evaluation function
# 3.5 store all above info

def site_preparations (site):
    # 3.1 import all tsfm for image sites
    if site["modality"] == "image":
        tsfm_path = DATASETS_FOLDER + "/{:s}/{:s}_tsfm.py".format(site["name"], site["name"])
        full_path = tsfm_path
        site["tsfm"] = load_transform_from_file(full_path)

    # 3.2 get the saved local head
    model_name  = site["name"]
    modality    = site["modality"]
    n_classes   = site["n_classes"]

    # overall best site head 
    saved_head  = SAVED_MODELS_FOLDER + "overall_best_local_heads" + model_name + ".pth" 
    site["trained_local_head"] = saved_head

    # 3.3 assign the correct local training function
    site["site_evaluation"] = assign_local_evaluation_function(site["modality"])

    return site

In [33]:
# 0. (BEFORE training) find all recorded models
# overall best local heads
local_head_folder = SAVED_MODELS_FOLDER + "/overall_best_local_heads"
# best global encoders
global_enc_pth = SAVED_MODELS_FOLDER + "/best_global_encoders.pth"

In [39]:
# 1. build global model
global_encoders = SharedEncoders(
    d_tabular   = D_TABULAR,
    d_embedding = D_EMBEDDING,
    d_fusion    = D_FUSION
    )
# dict to torch
global_state = torch.load(global_enc_pth, map_location="cpu")

# torch to model (BUT do we need the global model here?)
global_encoders.load_state_dict(global_state, strict = False)
print("Global encoders loaded!")

Global encoders loaded!


In [36]:
# 2. import all sites (basic info)
# with open("sites_info_tabular.json", "r") as f:
with open("sites_info.json", "r") as f:
    sites_raw = json.load(f)
sites_raw

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'Datasets/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'Datasets/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'Datasets/tabular_1/diabetes_012_test.csv'},
 {'name': 'image_1',
  'modality': 'image',
  'n_classes': 5,
  'clean_dataset_train': 'Datasets/image_1/train',
  'clean_dataset_val': 'Datasets/image_1/val',
  'clean_dataset_test': 'Datasets/image_1/test'}]

In [38]:
# 3. all sites preparations:
# FOR EACH SITE:
# 3.1 DATA: get testing dataset (DONE)
# 3.2 MODEL: find the head path (with site name)

sites = sites_raw.copy()
for site in sites_raw:
    site = site_preparations(site)

sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'label_col': 'Diabetes_012',
  'n_classes': 2,
  'clean_dataset_train': 'Datasets/tabular_1/diabetes_012_train.csv',
  'clean_dataset_val': 'Datasets/tabular_1/diabetes_012_val.csv',
  'clean_dataset_test': 'Datasets/tabular_1/diabetes_012_test.csv',
  'trained_local_head': 'SavedModelsoverall_best_local_headstabular_1.pth',
  'site_evaluation': <function SiteTrainingFunctions.tabular_site_evaluation.final_evaluation()>},
 {'name': 'image_1',
  'modality': 'image',
  'n_classes': 5,
  'clean_dataset_train': 'Datasets/image_1/train',
  'clean_dataset_val': 'Datasets/image_1/val',
  'clean_dataset_test': 'Datasets/image_1/test',
  'tsfm': Compose(
      Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
      RandomHorizontalFlip(p=0.5)
      RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
      ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=(0.9, 1.1), hue

In [ ]:
# 4. operate federated training loop
# 4.0 configs
# (MANUALLY) set training rounds
pass

# (AOTU) decide components to update
for site in sites:
    if not TABULAR_UPDATE:
        if site["modality"] == "tabular":
            TABULAR_UPDATE = True
            continue
        else: pass

    if not IMAGE_UPDATE:
        if site["modality"] == "image":
            IMAGE_UPDATE = True
            continue
        else: pass

    if not FUSION_UPDATE:
        if site["modality"] == "multi":
            FUSION_UPDATE = True
            continue
        else: pass

# 4.1 initialization -- best global: save the initial global state as current best
pass

# 4.2 initialization -- best evaluation scores
best_ckpt_eva_results = [] # list of dict: {"site_name", "val_loss", "val_acc", "num_samples"}
for site in sites:
    results = {
        "site_name": site["name"],
        "val_loss" : float("inf"),
        "val_acc"  : -1.0,
        "num_samples": 0
        }
    best_ckpt_eva_results.append(results)

In [ ]:
# 4.3 evaluation

# 1. one federated training round -> ALL sites FULLY trained ONCE
scores = complete_site_evaluation(global_state, sites)

global update round 1:
Start site traininig
tabular_1: tabular site training START
 [best updated] acc: 0.8342
 [best updated] acc: 0.8359
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.6392
 [best updated] acc: 0.6456
 [best updated] acc: 0.6603
 [best updated] acc: 0.6835
image_1: image site training DONE
Finish site traininig
Start head re-training
tabular_1: tabular site training START
 [best updated] acc: 0.8358
 [best updated] acc: 0.8360
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.6793
 [best updated] acc: 0.6772
image_1: image site training DONE
Finish head re-training

global update round 2:
Start site traininig
tabular_1: tabular site training START
 [best updated] acc: 0.8355
 [best updated] acc: 0.8356
 [best updated] acc: 0.8357
tabular_1: tabular site training DONE
image_1: image site training START
 [best updated] acc: 0.7004
 [best updated] acc: 0.7046
image_1: image sit

In [ ]:
# 4. final evaluation